### import libraries

In [1]:
import concurrent.futures
import requests
import concurrent.futures
import pandas as pd
import time
import calendar
import datetime


# Main API

Since we are fetching a lot of data we need to define a fast function that can be run asynchronically to fetch as much data as fast as possible.

In [16]:


def get_them_api(city, lat, lon, api_parameter, year=None):

    """
    Gets average monthly temperature for a City.
    Args:
        city (str): City name.
        lat (float): Latitude of the city.
        lon (float): Longitude of the city.
        api_parameter (str): API parameter for weather data.
        year (int): Year for which to retrieve data (default: current year)
    Returns:
        None
    """

    BASE_URL = "https://archive-api.open-meteo.com/v1/era5"
    #adds the weather data
    monthly_averages = []  # Renamed variable to store monthly averages
    # Loop through the months
    for month in range(1, 13):
        month_start = f"{year}-{month:02}-01"
        # Get the last day of the month of 30 days month
        if month in [4, 6, 9, 11]:
            month_end = f"{year}-{month:02}-30"
        # account for february
        elif month == 2:
            # if is leap year
            if calendar.isleap(year):
                month_end = f"{year}-{month:02}-29"
            # if not
            else:
                month_end = f"{year}-{month:02}-28"
        #for months with 31 days
        else:
            month_end = f"{year}-{month:02}-31"
        # format the API URL with latitude longitude dates and paramter
        url = f"{BASE_URL}?latitude={lat}&longitude={lon}&start_date={month_start}&end_date={month_end}&hourly={api_parameter}"
        response = requests.get(url)
        # if response is oke
        if response.status_code == 200:
            data = response.json()
            temps = data['hourly'][f"{api_parameter}"]
            # calculate the monthly average temperature
            if len(temps) > 0:
                monthly_avg = round(sum(temps) / len(temps), 2)
            # error handling if no response
            else:
                monthly_avg = None
            monthly_averages.append(monthly_avg)  # Append to the list

    # add the monthly average temperatures to the list monthly averages and then to the dict monthly
    monthly = {
        'city': city,
        'jan': monthly_averages[0],
        'feb': monthly_averages[1],
        'mar': monthly_averages[2],
        'apr': monthly_averages[3],
        'may': monthly_averages[4],
        'jun': monthly_averages[5],
        'jul': monthly_averages[6],
        'aug': monthly_averages[7],
        'sep': monthly_averages[8],
        'oct': monthly_averages[9],
        'nov': monthly_averages[10],
        'dec': monthly_averages[11],
    }
    # return all monthly averages of the given city
    return monthly


In [17]:
def go_getter(csv_files_list, api_parameter, merged_filename, year=None):
    """
    Fetches weather data for cities in multiple CSV files using concurrent.futures.
    multiple threads allow the get_them_api to run symoutanesly.

    Args:
        csv_files_list (list): List of CSV file paths.
        api_parameter (str): API parameter for weather data.
        merged_filename (str): Filename for the merged data.
        year (int): Year for which to fetch weather data. If not provided, the current year will be used.

    Returns:
        None
    """
    # Set the year to the current year if not provided
    if year is None:
        year = datetime.datetime.now().year
    # Loop through the CSV files (countries) and fetch weather data for each city in country
    for csv_file in csv_files_list:
        # Add a delay of 40 seconds before making the API request after every country
        time.sleep(40)
        # Read the CSV file
        df = pd.read_csv(csv_file)
        # Fetch weather data for each city using concurrent.futures for faster processing
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = [executor.submit(get_them_api, row['name'], row['latitude'], row['longitude'], api_parameter, year) for _, row in df.iterrows()]
            concurrent.futures.wait(results)
            data_list = [result.result() for result in results]
        # Transform data_list to a DataFrame
        df_weather = pd.DataFrame(data_list)
        # Merge the weather data with the original DataFrame
        df = pd.concat([df, df_weather], axis=1)
        # Modify the filename
        naming = csv_file.replace("Cities.csv", f"{merged_filename}.csv")
        # Save the merged data to the CSV file with modified filename
        df.to_csv(naming, index=False)


## ALL PARAMETERS that can be used for the go_getter function and the &hourly= parameter
#### see [Open-Meteo Documentation](https://open-meteo.com/en/docs) for more information


Parameter | Description
temperature_2m | Instant °C (°F) Air temperature at 2 meters above ground

### European Country Codes (ISO 3166-1 alpha-2) for file names replication

### Each run with the go_getter function takes roughly 40-45 mins for each year (70 mio api Calls) the sleep timer is set to 25 seconds to avoid api rate limit which also increases wait time

The go_getter function needs to run each time for every year/paramter combination to get the data
to make this more efficent we defined a for loop that loops through every year. So we can automate a tidouse process of renaming a file by doing everything in this loop. The output of this loop is a merged csv file in Capital letters followed by the year, the data was taken (the api lets us take date until 1960)

In [2]:
# replicate the csv files for each country which where createad with the geochache api
european_country_codes = ['AL', 'AD', 'AM', 'AT', 'AZ', 'BY', 'BE', 'BA', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FO', 'FI', 'FR', 'GE', 'DE', 'GI', 'GR', 'GL', 'GG', 'HU', 'IS', 'IE', 'IM', 'IT', 'JE', 'KZ', 'XK', 'LV', 'LT', 'LU', 'MK', 'MT', 'MD', 'MC', 'ME', 'NL', 'NO', 'PL', 'PT', 'RO', 'RU', 'RS', 'SK', 'SI', 'ES', 'SE', 'CH', 'UA', 'GB', 'TR', 'UZ']

csv_files = [code_ + "_Cities.csv" for code_ in european_country_codes]

csv_files

['AL_Cities.csv',
 'AD_Cities.csv',
 'AM_Cities.csv',
 'AT_Cities.csv',
 'AZ_Cities.csv',
 'BY_Cities.csv',
 'BE_Cities.csv',
 'BA_Cities.csv',
 'BG_Cities.csv',
 'HR_Cities.csv',
 'CY_Cities.csv',
 'CZ_Cities.csv',
 'DK_Cities.csv',
 'EE_Cities.csv',
 'FO_Cities.csv',
 'FI_Cities.csv',
 'FR_Cities.csv',
 'GE_Cities.csv',
 'DE_Cities.csv',
 'GI_Cities.csv',
 'GR_Cities.csv',
 'GL_Cities.csv',
 'GG_Cities.csv',
 'HU_Cities.csv',
 'IS_Cities.csv',
 'IE_Cities.csv',
 'IM_Cities.csv',
 'IT_Cities.csv',
 'JE_Cities.csv',
 'KZ_Cities.csv',
 'XK_Cities.csv',
 'LV_Cities.csv',
 'LT_Cities.csv',
 'LU_Cities.csv',
 'MK_Cities.csv',
 'MT_Cities.csv',
 'MD_Cities.csv',
 'MC_Cities.csv',
 'ME_Cities.csv',
 'NL_Cities.csv',
 'NO_Cities.csv',
 'PL_Cities.csv',
 'PT_Cities.csv',
 'RO_Cities.csv',
 'RU_Cities.csv',
 'RS_Cities.csv',
 'SK_Cities.csv',
 'SI_Cities.csv',
 'ES_Cities.csv',
 'SE_Cities.csv',
 'CH_Cities.csv',
 'UA_Cities.csv',
 'GB_Cities.csv',
 'TR_Cities.csv',
 'UZ_Cities.csv']

In [ ]:

#running this code takes roughly 4 to 5 hours

# define a list of years to loop through
#years = [2000,2013, 2003, 2018, 2020, 2021, 2022]
#years = [1970, 1980, 1990, 2003, 2013]
years = [2013]

# loop through each year
for year in years:
    # define the filename for the csv file based on the year
    csv_file_name = "Temp_" + str(year)[-2:]

    # run the go_getter function for the temperature_2m parameter
    go_getter(csv_files, "temperature_2m", csv_file_name, year=year)

    # create a list of all the csv files for the current year (reproduce the csv files)
    csv_files_temp = ["{}_Temp_{}.csv".format(code, str(year)[-2:]) for code in european_country_codes]

    # merge all the csv files for the current year into one dataframe
    all_temp = pd.concat([pd.read_csv(f) for f in csv_files_temp])

    # remove the 'name' column because cityname is in "city"
    all_temp = all_temp.drop(columns=['name'])

    # remove duplicates
    all_temp = all_temp.drop_duplicates()

    # save the merged dataframe to a csv file for the current year
    all_temp.to_csv('TEMP_{}.csv'.format(str(year)[-2:]), index=False)

    # add a time delay of 8 min  before moving on to the next year to avoid network error
    time.sleep(480)
